In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class WaterQualityClassifier:
    def __init__(self):
        self.quality_labels = {0: 'green', 1: 'yellow', 2: 'orange', 3: 'darkred'}
        self.color_map = {0: 'green', 1: 'yellow', 2: 'orange', 3: 'darkred'}
        
        # Default weights (modifiable)
        self.weights = {
            'DO_mg': 0.25,
            'TSS': 0.20, 
            'Sal': 0.05,
            'N_TOTAL': 0.20,
            'P_PO4': 0.15,
            'P_TOTAL': 0.15
        }
        
        # Level ranges - based on stricter thresholds
        self.level_ranges = [
            (0, 0.4, 0),      # Green: Excellent
            (0.4, 1.0, 1),    # Yellow: Good  
            (1.0, 1.74, 2),   # Orange: Fair
            (1.74, 4, 3)      # Dark Red: Poor
        ]
    
    def load_and_merge_data(self, metadata_path, quality_data_path):
        """Load and merge site metadata and water quality data"""
        metadata = pd.read_csv(metadata_path)
        quality_data = pd.read_csv(quality_data_path)
        
        # Merge to add latitude and longitude
        merged_data = quality_data.merge(
            metadata[['site_id', 'latitude', 'longitude']], 
            on='site_id', 
            how='left'
        )
        
        # Convert date format
        merged_data['date'] = pd.to_datetime(merged_data['date'])
        
        return metadata, quality_data, merged_data
    
    def score_single_parameter(self, value, parameter):
        """Score a single parameter (0-3), does not handle NaN"""
        if parameter == 'DO_mg':
            # ≥8(0), 6-8(1), 4-6(2), <4(3)
            if value >= 8:
                return 0
            elif value >= 6:
                return 1
            elif value >= 4:
                return 2
            else:
                return 3
                
        elif parameter == 'TSS':
            # <5(0), 5-20(1), 20-50(2), >50(3)
            if value < 5:
                return 0
            elif value <= 20:
                return 1
            elif value <= 50:
                return 2
            else:
                return 3
                
        elif parameter == 'Sal':
            # 30-38(0), 25-30 or 38-41(1), 20-25 or 41-44(2), others(3)
            if 30 <= value <= 38:
                return 0
            elif (25 <= value < 30) or (38 < value <= 41):
                return 1
            elif (20 <= value < 25) or (41 < value <= 44):
                return 2
            else:
                return 3
                
        elif parameter in ['N_TOTAL', 'P_PO4', 'P_TOTAL']:
            # Convert from mg/L to μg/L (×1000)
            value_ug_per_l = value * 1000
            
            if parameter == 'N_TOTAL':
                # <150(0), 150-300(1), 300-600(2), >600(3)
                if value_ug_per_l < 150:
                    return 0
                elif value_ug_per_l <= 300:
                    return 1
                elif value_ug_per_l <= 600:
                    return 2
                else:
                    return 3
                    
            elif parameter == 'P_PO4':
                # <30(0), 30-60(1), 60-120(2), >120(3)
                if value_ug_per_l < 30:
                    return 0
                elif value_ug_per_l <= 60:
                    return 1
                elif value_ug_per_l <= 120:
                    return 2
                else:
                    return 3
                    
            elif parameter == 'P_TOTAL':
                # <45(0), 45-90(1), 90-180(2), >180(3)
                if value_ug_per_l < 45:
                    return 0
                elif value_ug_per_l <= 90:
                    return 1
                elif value_ug_per_l <= 180:
                    return 2
                else:
                    return 3
        
        return None
    
    def calculate_weighted_score(self, row):
        """Calculate weighted average score"""
        parameters = ['DO_mg', 'TSS', 'Sal', 'N_TOTAL', 'P_PO4', 'P_TOTAL']
        
        weighted_sum = 0
        valid_weight_sum = 0
        param_scores = {}
        
        for param in parameters:
            if param in row and pd.notna(row[param]):
                score = self.score_single_parameter(row[param], param)
                if score is not None:
                    param_scores[param] = score
                    weight = self.weights.get(param, 0)
                    weighted_sum += score * weight
                    valid_weight_sum += weight
        
        if valid_weight_sum > 0:
            weighted_average = weighted_sum / valid_weight_sum
            return weighted_average, param_scores
        else:
            return None, param_scores
    
    def score_to_level(self, score):
        """Map continuous score to discrete level"""
        if score is None:
            return None
        
        for min_val, max_val, level in self.level_ranges:
            if min_val <= score < max_val:
                return level
        
        return 3  # fallback
    
    def classify_water_quality(self, merged_data):
        """Classify water quality"""
        data = merged_data.copy()
        
        print("Classifying water quality...")
        
        scores_and_params = data.apply(self.calculate_weighted_score, axis=1)
        
        data['water_quality_score'] = [item[0] for item in scores_and_params]
        param_scores_list = [item[1] for item in scores_and_params]
        
        param_score_df = pd.DataFrame(param_scores_list)
        param_score_df.columns = [f'{col}_score' for col in param_score_df.columns]
        
        data = pd.concat([data, param_score_df], axis=1)
        
        data['quality_level'] = data['water_quality_score'].apply(self.score_to_level)
        data['quality_label'] = data['quality_level'].map(self.quality_labels)
        
        data['water_quality_simple'] = data['quality_level'].apply(
            lambda x: 'good' if x in [0, 1, 2] else ('bad' if x == 3 else None)
        )
        
        return data
    
    def calculate_site_recent_averages(self, classified_data):
        """Calculate site averages over the past year"""
        print("Calculating recent one-year averages for each site...")
        
        site_averages = []
        
        for site_id in classified_data['site_id'].unique():
            site_data = classified_data[classified_data['site_id'] == site_id].copy()
            
            latest_date = site_data['date'].max()
            cutoff_date = latest_date - timedelta(days=365)
            
            recent_data = site_data[site_data['date'] >= cutoff_date]
            
            if len(recent_data) == 0:
                continue
            
            parameters = ['DO_mg', 'TSS', 'Sal', 'N_TOTAL', 'P_PO4', 'P_TOTAL']
            avg_row = {}
            
            avg_row['site_id'] = site_id
            avg_row['site_name_short'] = recent_data['site_name_short'].iloc[0]
            avg_row['latitude'] = recent_data['latitude'].iloc[0]
            avg_row['longitude'] = recent_data['longitude'].iloc[0]
            
            avg_row['latest_date'] = latest_date
            avg_row['cutoff_date'] = cutoff_date
            avg_row['records_count'] = len(recent_data)
            
            for param in parameters:
                if param in recent_data.columns:
                    avg_row[f'{param}_mean'] = recent_data[param].mean()
                else:
                    avg_row[f'{param}_mean'] = None
            
            avg_values = {param: avg_row[f'{param}_mean'] for param in parameters}
            weighted_score, param_scores = self.calculate_weighted_score(avg_values)
            
            avg_row['avg_water_quality_score'] = weighted_score
            avg_row['avg_quality_level'] = self.score_to_level(weighted_score)
            
            if avg_row['avg_quality_level'] is not None:
                avg_row['avg_quality_label'] = self.quality_labels[avg_row['avg_quality_level']]
            else:
                avg_row['avg_quality_label'] = 'Unclassified'
            
            for param, score in param_scores.items():
                avg_row[f'{param}_avg_score'] = score
            
            if avg_row['avg_quality_level'] is not None:
                avg_row['avg_water_quality_simple'] = (
                    'good' if avg_row['avg_quality_level'] in [0, 1, 2] else 'bad'
                )
            else:
                avg_row['avg_water_quality_simple'] = None
            
            site_averages.append(avg_row)
        
        return pd.DataFrame(site_averages)
    
    def print_statistics(self, classified_data, avg_data):
        """Print statistics summary"""
        print("\n" + "="*50)
        print("Water Quality Classification Report")
        print("="*50)
        
        total_records = len(classified_data)
        valid_scores = classified_data['water_quality_score'].dropna()
        
        print(f"\n[Overall]")
        print(f"Total records: {total_records:,}")
        print(f"Valid scored records: {len(valid_scores):,}")
        print(f"Unscored records: {total_records - len(valid_scores):,}")
        
        if len(valid_scores) > 0:
            print(f"Mean water quality score: {valid_scores.mean():.3f}")
            print(f"Median score: {valid_scores.median():.3f}")
            print(f"Score range: {valid_scores.min():.3f} ~ {valid_scores.max():.3f}")
        
        print(f"\n[Level Distribution]")
        level_counts = classified_data['quality_level'].value_counts().sort_index()
        for level in [0, 1, 2, 3]:
            count = level_counts.get(level, 0)
            if len(valid_scores) > 0:
                pct = count / len(valid_scores) * 100
                print(f"  {self.quality_labels[level]}: {count:,} records ({pct:.1f}%)")
        
        if len(valid_scores) > 0:
            print(f"\n[Score Range Distribution]")
            ranges = [
                (0, 0.4, 'Green'),
                (0.4, 1.0, 'Yellow'),
                (1.0, 1.74, 'Orange'),
                (1.74, 4, 'Dark Red')
            ]
            
            for min_val, max_val, color in ranges:
                if max_val == 4:
                    count = sum(valid_scores >= min_val)
                else:
                    count = sum((valid_scores >= min_val) & (valid_scores < max_val))
                pct = count / len(valid_scores) * 100
                print(f"  {min_val:.2f}-{max_val:.2f} ({color}): {count:,} records ({pct:.1f}%)")
        
        if not avg_data.empty:
            print(f"\n[Site Averages]")
            print(f"Number of sites: {len(avg_data)}")
            
            valid_avg_scores = avg_data['avg_water_quality_score'].dropna()
            if len(valid_avg_scores) > 0:
                print(f"Mean of site average scores: {valid_avg_scores.mean():.3f}")
                
                print(f"\nRecent one-year average classification per site:")
                for _, row in avg_data.iterrows():
                    if row['avg_quality_level'] is not None:
                        print(f"  {row['site_name_short']}: {row['avg_quality_label']} " +
                              f"(Score: {row['avg_water_quality_score']:.3f}, Records: {row['records_count']})")
                    else:
                        print(f"  {row['site_name_short']}: Unclassified (Records: {row['records_count']})")
    
    def visualize_results(self, classified_data, avg_data):
        """Visualize results"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        plt.suptitle('Port Phillip Bay Water Quality Analysis Report', fontsize=16, fontweight='bold')
        
        # 1. Quality level distribution (all records)
        valid_data = classified_data.dropna(subset=['quality_level'])
        if not valid_data.empty:
            quality_counts = valid_data['quality_level'].value_counts().sort_index()
            colors = [self.color_map[int(i)] for i in quality_counts.index]
            
            bars = axes[0,0].bar(range(len(quality_counts)), quality_counts.values, color=colors)
            axes[0,0].set_title('Water Quality Level Distribution (All Records)')
            axes[0,0].set_xlabel('Quality Level')
            axes[0,0].set_ylabel('Record Count')
            axes[0,0].set_xticks(range(len(quality_counts)))
            axes[0,0].set_xticklabels([self.quality_labels[int(i)] for i in quality_counts.index], rotation=45)
            
            for bar in bars:
                height = bar.get_height()
                axes[0,0].text(bar.get_x() + bar.get_width()/2., height,
                              f'{int(height)}', ha='center', va='bottom')
        
        # 2. Average site quality level distribution
        if not avg_data.empty:
            valid_avg = avg_data.dropna(subset=['avg_quality_level'])
            if not valid_avg.empty:
                avg_quality_counts = valid_avg['avg_quality_level'].value_counts().sort_index()
                colors_avg = [self.color_map[int(i)] for i in avg_quality_counts.index]
                
                bars = axes[0,1].bar(range(len(avg_quality_counts)), avg_quality_counts.values, color=colors_avg)
                axes[0,1].set_title('Average Site Quality Level Distribution')
                axes[0,1].set_xlabel('Quality Level')
                axes[0,1].set_ylabel('Site Count')
                axes[0,1].set_xticks(range(len(avg_quality_counts)))
                axes[0,1].set_xticklabels([self.quality_labels[int(i)] for i in avg_quality_counts.index], rotation=45)
                
                for bar in bars:
                    height = bar.get_height()
                    axes[0,1].text(bar.get_x() + bar.get_width()/2., height,
                                  f'{int(height)}', ha='center', va='bottom')
        
        # 3. Histogram of scores
        valid_scores = classified_data['water_quality_score'].dropna()
        if len(valid_scores) > 0:
            axes[0,2].hist(valid_scores, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
            axes[0,2].axvline(x=0.4, color='gold', linestyle='--', linewidth=2, label='Green→Yellow (0.4)')
            axes[0,2].axvline(x=1.0, color='orange', linestyle='--', linewidth=2, label='Yellow→Orange (1.0)')
            axes[0,2].axvline(x=1.74, color='red', linestyle='--', linewidth=2, label='Orange→Dark Red (1.74)')
            axes[0,2].set_title('Water Quality Score Distribution')
            axes[0,2].set_xlabel('Score')
            axes[0,2].set_ylabel('Frequency')
            axes[0,2].legend()
            axes[0,2].grid(True, alpha=0.3)
        
        # 4. Time series trend
        if 'date' in classified_data.columns:
            monthly_data = classified_data.dropna(subset=['water_quality_score'])
            if not monthly_data.empty:
                monthly_score = monthly_data.groupby(monthly_data['date'].dt.to_period('M'))['water_quality_score'].mean()
                axes[1,0].plot(range(len(monthly_score)), monthly_score.values, marker='o', linewidth=2, markersize=4)
                axes[1,0].set_title('Water Quality Score Time Trend')
                axes[1,0].set_xlabel('Time')
                axes[1,0].set_ylabel('Average Score')
                axes[1,0].grid(True, alpha=0.3)
                
                if len(monthly_score) > 10:
                    step = len(monthly_score) // 8
                    ticks = range(0, len(monthly_score), step)
                    labels = [str(monthly_score.index[i])[:7] for i in ticks]
                    axes[1,0].set_xticks(ticks)
                    axes[1,0].set_xticklabels(labels, rotation=45)
        
        # 5. Boxplot of parameter scores
        param_score_cols = [col for col in classified_data.columns if col.endswith('_score') and 'water_quality' not in col]
        if param_score_cols:
            box_data = []
            labels = []
            for col in param_score_cols:
                data = classified_data[col].dropna()
                if len(data) > 0:
                    box_data.append(data.values)
                    labels.append(col.replace('_score', ''))
            
            if box_data:
                bp = axes[1,1].boxplot(box_data, labels=labels, patch_artist=True)
                axes[1,1].set_title('Distribution of Parameter Scores')
                axes[1,1].tick_params(axis='x', rotation=45)
                axes[1,1].set_ylabel('Score (0-3)')
                axes[1,1].grid(True, alpha=0.3)
                
                colors = ['lightblue', 'lightgreen', 'lightcoral', 'lightyellow', 'lightpink', 'lightgray']
                for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                    patch.set_facecolor(color)
        
        # 6. Site average score comparison
         if not avg_data.empty:
            valid_sites = avg_data.dropna(subset=['avg_water_quality_score'])
            if not valid_sites.empty:
                # order
                valid_sites_sorted = valid_sites.sort_values('avg_water_quality_score')
                
                bars = axes[1,2].bar(range(len(valid_sites_sorted)), 
                                   valid_sites_sorted['avg_water_quality_score'], 
                                   color='steelblue', alpha=0.7)
                axes[1,2].set_title('Average water quality scores at each site')
                axes[1,2].set_xlabel('Monitoring sites')
                axes[1,2].set_ylabel('Average water quality score')
                axes[1,2].set_xticks(range(len(valid_sites_sorted)))
                axes[1,2].set_xticklabels(valid_sites_sorted['site_name_short'], rotation=45)
                axes[1,2].grid(True, alpha=0.3)
                
                # 添加等级分界线
                axes[1,2].axhline(y=0.4, color='gold', linestyle='--', alpha=0.7, label='yellow_line')
                axes[1,2].axhline(y=1.0, color='orange', linestyle='--', alpha=0.7, label='orange_line')
                axes[1,2].axhline(y=1.74, color='red', linestyle='--', alpha=0.7, label='darkred_line')
                axes[1,2].legend()
        
        plt.tight_layout()
        plt.show()
    
    def run_analysis(self, metadata_path, quality_data_path):
        """run complete analysis"""
        print("="*60)
        print("data analysis")
        print("="*60)
        
        # 1.load and combine data
        print("\n loading the data...")
        metadata, quality_data, merged_data = self.load_and_merge_data(metadata_path, quality_data_path)
        print(f" data loading complete")
        print(f"   - Site metadata: {len(metadata)} ")
        print(f"   - Water quality monitoring data: {len(quality_data):,} records")
        print(f"   - Data time span: {merged_data['date'].min().strftime('%Y-%m-%d')} to {merged_data['date'].max().strftime('%Y-%m-%d')}")
        
        # 2. Water quality classification
        print("\n Water quality classification...")
        classified_data = self.classify_water_quality(merged_data)
        print(" Water quality grading completed")
        
        # 3. Calculate site averages
        print("\n Calculating the average data of each site in the past year...")
        avg_data = self.calculate_site_recent_averages(classified_data)
        print(f" The average data of the site is calculated and processed {len(avg_data)} sites")
        
        # 4. print
        self.print_statistics(classified_data, avg_data)
        
        # 5. visualization
        print(f"\n 正在生成可视化图表...")
        self.visualize_results(classified_data, avg_data)
        print(" 图表生成完成")
        
        # 6. 
        print(f"\n Saving results...")
        classified_data.to_csv('water_quality_classified_lll.csv', index=False, encoding='utf-8-sig')
        if not avg_data.empty:
            avg_data.to_csv('site_averages_recent_year_lll.csv', index=False, encoding='utf-8-sig')
        
        print(" Results saved:")
        print(" - water_quality_classified.csv: Contains the water quality score and grade for each record")
        print(" - site_averages_recent_year.csv: Average water quality data for each site in the most recent year")
        print(" - Both files contain a 'water_quality_simple' column (good/bad judgment)")
        
        # Display statistics for simplified judgments
        if 'water_quality_simple' in classified_data.columns:
            simple_counts = classified_data['water_quality_simple'].value_counts()
            print(f"   - Overall water quality assessment: Good: {simple_counts.get('good', 0):,} , Bad: {simple_counts.get('bad', 0):,} ")
        
        if not avg_data.empty and 'avg_water_quality_simple' in avg_data.columns:
            avg_simple_counts = avg_data['avg_water_quality_simple'].value_counts()
            print(f"   - Site water quality assessment: Good: {avg_simple_counts.get('good', 0)} 个, Bad: {avg_simple_counts.get('bad', 0)} 个站点")
        
        print(f"\n 分析完成！")
        
        return {
            'classified_data': classified_data,
            'site_averages': avg_data,
            'metadata': metadata
        }

# 
if __name__ == "__main__":
    # create analyser
    classifier = WaterQualityClassifier()
    
    # 可选：change weight
    # classifier.weights = {
    #     'DO_mg': 0.30,      # 提高溶解氧权重
    #     'TSS': 0.25,        # 提高悬浮物权重  
    #     'Sal': 0.05,        # 保持盐度低权重
    #     'N_TOTAL': 0.15,    # 降低总氮权重
    #     'P_PO4': 0.10,      # 降低磷酸盐权重
    #     'P_TOTAL': 0.15     # 保持总磷权重
    # }
    
    try:
        # 运行分析
        results = classifier.run_analysis(
            metadata_path='site_metadata.csv',
            quality_data_path='cleaned_water_quality_data.csv'
        )
        
        # 显示部分结果
        print(f"\n Preview of water quality scores for the first 10 records:")
        preview_cols = ['site_name_short', 'date', 'water_quality_score', 'quality_label', 'water_quality_simple']
        preview_data = results['classified_data'][preview_cols].head(10)
        print(preview_data.to_string(index=False))
        
    except FileNotFoundError as e:
        print(f" file not found: {e}")
        print("Please make sure the following files are in the current directory:")
        print("  - site_metadata_lll.csv")
        print("  - cleaned_water_quality_data_lll.csv")
    except Exception as e:
        print(f" Error at runtime: {e}")
        import traceback
        traceback.print_exc()